# 2 · Estrazione EMBEDDING dei foundation model

Per ogni TC, 7 reti neurali **pre-addestrate e congelate** (backbone) producono un vettore di
numeri (embedding). Salviamo un file per modello in `cache_features/feat_<modello>.parquet`.

> ⚠️ **Kernel: "PANORAMA ML (.venv-ml)"** · GPU consigliata.
> ♻️ **Resumabile** e a modelli **isolati**: se un modello non è disponibile, viene saltato.
> Tutta la logica (ROI, architetture, loader) è **qui dentro**.

In [1]:
# --- Setup ---

# --- Setup, percorsi e RIPRODUCIBILITÀ --------------------------------------
import os, warnings, random
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED); random.seed(SEED); np.random.seed(SEED)

# trova la radice del progetto (contiene imagesTr/ e il file clinico)
CWD = Path.cwd()
def _e_radice(p): return (p / "imagesTr").exists() and (p / "cache" / "clinical_information.xlsx").exists()
PROJECT_DIR = next((p for p in [CWD, *CWD.parents] if _e_radice(p)), CWD)
IMAGES_DIR = PROJECT_DIR / "imagesTr"
LABELS_DIR = PROJECT_DIR / "labelsTr"
CLINICAL   = PROJECT_DIR / "cache" / "clinical_information.xlsx"
CACHE_FEAT = PROJECT_DIR / "ml_study" / "cache_features"; CACHE_FEAT.mkdir(parents=True, exist_ok=True)

# Costanti delle maschere PANORAMA: 0=sfondo,1=lesione,2=vene,3=arterie,4=pancreas,5=dotto,6=coledoco
LESION_CLASS, PANCREAS_CLASS = 1, 4
print("Radice progetto:", PROJECT_DIR, "| immagini:", IMAGES_DIR.exists(), "| clinico:", CLINICAL.exists())

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", DEVICE)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

Radice progetto: D:\PANORAMA_v2 | immagini: True | clinico: True


PyTorch 2.6.0+cu124 | device: cuda


## 1) La coorte (326 casi)
Teniamo i casi **scaricati** con **sesso ed età** presenti (esclude i sottoinsiemi pubblici
NIH/MSD privi di quei metadati → restano i centri olandesi RUMC + UMCG).
- `y` = 1 se PDAC, 0 se controllo · `sex_bin` = 0 F / 1 M · `age` in anni (da `'077Y'` a 77).

In [2]:
df = pd.read_excel(CLINICAL).rename(columns={"PANORAMA_study_id": "study_id"})
df["age"] = pd.to_numeric(df["patient_age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
df["sex_bin"] = df["patient_sex"].astype(str).str.strip().str.upper().str[0].map({"F": 0, "M": 1})
df["y"] = (df["label"].astype(str).str.upper() == "PDAC").astype(int)
df["ha_img"] = df["study_id"].map(lambda s: (IMAGES_DIR / f"{s}_0000.nii.gz").exists())
cohort = df[df["ha_img"] & df["sex_bin"].notna() & df["age"].notna()][
    ["study_id", "y", "age", "sex_bin", "label", "level"]].reset_index(drop=True)
print("Casi nella coorte:", len(cohort))
print(cohort["y"].map({1: "PDAC", 0: "controllo"}).value_counts().to_string())
cohort.head()

Casi nella coorte: 326
y
PDAC         182
controllo    144


,study_id,y,age,sex_bin,label,level
0,100012_00001,0,73.0,1.0,non-PDAC,radiology
1,100030_00001,1,51.0,0.0,PDAC,histopathology
2,100031_00001,0,38.0,1.0,non-PDAC,radiology
3,100037_00001,0,23.0,1.0,non-PDAC,radiology
4,100043_00001,1,73.0,0.0,PDAC,pathology


## 2) La ROI del pancreas
I modelli 3D lavorano su un **ritaglio** attorno al pancreas: prendiamo il *bounding box* della
maschera (pancreas + lesione) e aggiungiamo un margine (25 mm) — perché i segni secondari del
PDAC (dotto dilatato, atrofia) stanno appena fuori dal parenchima. `clip_ct` finestra i valori HU.

In [3]:
import SimpleITK as sitk
ROI_MARGIN_MM = 25.0

def carica_img_mask(sid):
    img = sitk.ReadImage(str(IMAGES_DIR / f"{sid}_0000.nii.gz"))
    spacing = img.GetSpacing()                        # (x,y,z) mm
    img_arr = sitk.GetArrayFromImage(img).astype(np.float32)   # (z,y,x)
    mp = LABELS_DIR / f"{sid}.nii.gz"
    mask = sitk.GetArrayFromImage(sitk.ReadImage(str(mp))).astype(np.int16) if mp.exists() else None
    if mask is not None and mask.shape != img_arr.shape: mask = None
    return img_arr, mask, spacing

def bbox_pancreas(mask, spacing, margin_mm=ROI_MARGIN_MM):
    target = (mask == PANCREAS_CLASS) | (mask == LESION_CLASS)
    if not target.any(): return None
    sx, sy, sz = spacing
    mvox = (int(round(margin_mm/sz)), int(round(margin_mm/sy)), int(round(margin_mm/sx)))
    coords = np.where(target); sl = []
    for c, mv, n in zip(coords, mvox, target.shape):
        sl.append(slice(max(int(c.min())-mv, 0), min(int(c.max())+mv+1, n)))
    return tuple(sl)

def extract_roi(sid):
    img, mask, spacing = carica_img_mask(sid)
    if mask is None: return dict(img=img, mask=None, spacing=spacing, used_mask=False)
    bb = bbox_pancreas(mask, spacing)
    if bb is None: return dict(img=img, mask=mask, spacing=spacing, used_mask=False)
    return dict(img=img[bb], mask=mask[bb], spacing=spacing, used_mask=True)

def clip_ct(a, lo=-100.0, hi=240.0):
    x = np.clip(a, lo, hi); return (x - lo) / (hi - lo)     # finestra addome -> [0,1]
print("Funzioni ROI pronte.")

Funzioni ROI pronte.


## 3) Architetture ricostruite (Models Genesis, STU-Net)
Due modelli non hanno un pacchetto pip: ne **ricostruiamo l'architettura** (dal sorgente
ufficiale) per poter caricare i pesi. Usiamo solo l'**encoder** (poi global-pooling del bottleneck).

In [4]:
# --- Models Genesis: 3D U-Net (sorgente ufficiale MrGiovanni/ModelsGenesis) ---

import torch.nn as nn, torch.nn.functional as F
class ContBatchNorm3d(nn.modules.batchnorm._BatchNorm):
    def _check_input_dim(self, i):
        if i.dim() != 5: raise ValueError("expected 5D input")
    def forward(self, i):
        self._check_input_dim(i)
        return F.batch_norm(i, self.running_mean, self.running_var, self.weight, self.bias, True, self.momentum, self.eps)
class LUConv(nn.Module):
    def __init__(self, ic, oc, act):
        super().__init__(); self.conv1 = nn.Conv3d(ic, oc, 3, padding=1); self.bn1 = ContBatchNorm3d(oc)
        self.activation = {"relu": nn.ReLU(oc), "prelu": nn.PReLU(oc), "elu": nn.ELU(inplace=True)}[act]
    def forward(self, x): return self.activation(self.bn1(self.conv1(x)))
def _make_nConv(ic, depth, act, double=False):
    if double: return nn.Sequential(LUConv(ic, 32*2**(depth+1), act), LUConv(32*2**(depth+1), 32*2**(depth+1), act))
    return nn.Sequential(LUConv(ic, 32*2**depth, act), LUConv(32*2**depth, 32*2**depth*2, act))
class DownTransition(nn.Module):
    def __init__(self, ic, depth, act):
        super().__init__(); self.ops = _make_nConv(ic, depth, act); self.maxpool = nn.MaxPool3d(2); self.current_depth = depth
    def forward(self, x):
        if self.current_depth == 3: out = self.ops(x); return out, out
        b = self.ops(x); return self.maxpool(b), b
class UNet3D(nn.Module):
    """Solo l'encoder ci serve (down_tr64->128->256->512). Il resto c'è per caricare i pesi."""
    def __init__(self, n_class=1, act="relu"):
        super().__init__()
        self.down_tr64 = DownTransition(1, 0, act); self.down_tr128 = DownTransition(64, 1, act)
        self.down_tr256 = DownTransition(128, 2, act); self.down_tr512 = DownTransition(256, 3, act)
        self.up_tr256 = _UpT(512, 512, 2, act); self.up_tr128 = _UpT(256, 256, 1, act)
        self.up_tr64 = _UpT(128, 128, 0, act); self.out_tr = _OutT(64, n_class)
    def forward(self, x): return x   # non usato: chiamiamo direttamente gli down_tr*
class _UpT(nn.Module):
    def __init__(self, ic, oc, depth, act):
        super().__init__(); self.up_conv = nn.ConvTranspose3d(ic, oc, 2, stride=2); self.ops = _make_nConv(ic+oc//2, depth, act, True)
    def forward(self, x, s): return self.ops(torch.cat((self.up_conv(x), s), 1))
class _OutT(nn.Module):
    def __init__(self, ic, n): super().__init__(); self.final_conv = nn.Conv3d(ic, n, 1); self.sigmoid = nn.Sigmoid()
    def forward(self, x): return self.sigmoid(self.final_conv(x))
print("Architettura Models Genesis (UNet3D) definita.")

Architettura Models Genesis (UNet3D) definita.


In [5]:
# --- STU-Net: architettura nnU-Net-style (patchata per girare senza il pacchetto nnunet) ---

class _InitHe:                    # stub (non serve: carichiamo pesi pretrained)
    def __init__(self,*a,**k): pass
    def __call__(self,m): pass
class BasicResBlock(nn.Module):
    def __init__(self, ic, oc, k=3, pad=1, stride=1, use_1x1=False):
        super().__init__()
        self.conv1 = nn.Conv3d(ic, oc, k, stride=stride, padding=pad); self.norm1 = nn.InstanceNorm3d(oc, affine=True); self.act1 = nn.LeakyReLU(inplace=True)
        self.conv2 = nn.Conv3d(oc, oc, k, padding=pad); self.norm2 = nn.InstanceNorm3d(oc, affine=True); self.act2 = nn.LeakyReLU(inplace=True)
        self.conv3 = nn.Conv3d(ic, oc, 1, stride=stride) if use_1x1 else None
    def forward(self, x):
        y = self.act1(self.norm1(self.conv1(x))); y = self.norm2(self.conv2(y))
        if self.conv3 is not None: x = self.conv3(x)
        return self.act2(y + x)
class STUNet(nn.Module):
    """nnU-Net-style. Usiamo encode() = solo encoder fino al bottleneck (512 canali)."""
    def __init__(self, input_channels, num_classes, depth=[1]*6, dims=[32,64,128,256,512,512],
                 pool_op_kernel_sizes=None, conv_kernel_sizes=None):
        super().__init__()
        pad = [[i//2 for i in k] for k in conv_kernel_sizes]; npool = len(pool_op_kernel_sizes)
        self.conv_blocks_context = nn.ModuleList()
        self.conv_blocks_context.append(nn.Sequential(
            BasicResBlock(input_channels, dims[0], conv_kernel_sizes[0], pad[0], use_1x1=True),
            *[BasicResBlock(dims[0], dims[0], conv_kernel_sizes[0], pad[0]) for _ in range(depth[0]-1)]))
        for d in range(1, npool+1):
            self.conv_blocks_context.append(nn.Sequential(
                BasicResBlock(dims[d-1], dims[d], conv_kernel_sizes[d], pad[d], stride=pool_op_kernel_sizes[d-1], use_1x1=True),
                *[BasicResBlock(dims[d], dims[d], conv_kernel_sizes[d], pad[d]) for _ in range(depth[d]-1)]))
        # (decoder definito solo per poter caricare TUTTI i pesi con strict=False)
        self.upsample_layers = nn.ModuleList([_STUp(dims[-1-u], dims[-2-u], pool_op_kernel_sizes[-1-u]) for u in range(npool)])
        self.conv_blocks_localization = nn.ModuleList()
        for u in range(npool):
            self.conv_blocks_localization.append(nn.Sequential(
                BasicResBlock(dims[-2-u]*2, dims[-2-u], conv_kernel_sizes[-2-u], pad[-2-u], use_1x1=True),
                *[BasicResBlock(dims[-2-u], dims[-2-u], conv_kernel_sizes[-2-u], pad[-2-u]) for _ in range(depth[-2-u]-1)]))
        self.seg_outputs = nn.ModuleList([nn.Conv3d(dims[-2-ds], num_classes, 1) for ds in range(npool)])
    def encode(self, x):
        for d in range(len(self.conv_blocks_context)-1): x = self.conv_blocks_context[d](x)
        return self.conv_blocks_context[-1](x)      # bottleneck (512 canali)
class _STUp(nn.Module):
    def __init__(self, ic, oc, pk): super().__init__(); self.conv = nn.Conv3d(ic, oc, 1); self.pk = pk
    def forward(self, x): return self.conv(nn.functional.interpolate(x, scale_factor=self.pk, mode="nearest"))
print("Architettura STU-Net definita.")

Architettura STU-Net definita.


## 4) I loader dei 7 modelli (backbone CONGELATI)
Ogni funzione costruisce un modello pre-addestrato e restituisce `embed(sid)` che dà il vettore.
- **2.5D** (ImageNet, RadImageNet): backbone 2D su fette assiali, poi media.
- **3D** (Merlin, CT-FM, MedicalNet, Genesis, STU-Net): encoder 3D sul volume/ROI.

In [6]:
# --- pipeline 2.5D condivisa (ImageNet, RadImageNet) ---
def slices_25d(sid, size=224, max_slices=24):
    """Fette assiali del pancreas -> tensore [S,3,224,224] (finestra HU, RGB, norm. ImageNet)."""
    r = extract_roi(sid); img = clip_ct(r["img"]); mask = r["mask"]
    counts = ((mask == PANCREAS_CLASS) if mask is not None else (img > 0)).reshape(img.shape[0], -1).sum(1)
    sel = [z for z in np.argsort(counts)[::-1] if counts[z] > 0][:max_slices] or [img.shape[0]//2]
    sl = torch.from_numpy(np.ascontiguousarray(img[sel])).float().unsqueeze(1)
    sl = torch.nn.functional.interpolate(sl, size=(size, size), mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
    mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1); std = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)
    return (sl - mean) / std
def embed_25d(net, sid, device, batch=32):
    x = slices_25d(sid).to(device); outs = []
    with torch.no_grad():
        for i in range(0, x.shape[0], batch): outs.append(net(x[i:i+batch]).cpu())
    return torch.cat(outs, 0).mean(0).numpy().astype(np.float32)   # media sulle fette
print("Helper 2.5D pronti.")

Helper 2.5D pronti.


In [7]:
import torchvision
from huggingface_hub import hf_hub_download

def build_imagenet(device):
    net = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    net.fc = torch.nn.Identity(); net.eval().to(device)
    for p in net.parameters(): p.requires_grad_(False)
    return lambda sid: embed_25d(net, sid, device)

def build_radimagenet(device):
    p = hf_hub_download(repo_id="Lab-Rasool/RadImageNet", filename="ResNet50.pt")
    sd = torch.load(p, map_location="cpu", weights_only=False); sd = sd.get("state_dict", sd)
    r = torchvision.models.resnet50(weights=None)
    net = torch.nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3, r.layer4,
                              torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten())
    net.load_state_dict({k[len("backbone."):]: v for k, v in sd.items() if k.startswith("backbone.")}, strict=False)
    net.eval().to(device)
    for p_ in net.parameters(): p_.requires_grad_(False)
    return lambda sid: embed_25d(net, sid, device)

def build_medicalnet(device):
    from monai.networks.nets import resnet50 as monai_resnet50
    net = monai_resnet50(pretrained=True, spatial_dims=3, n_input_channels=1, feed_forward=False,
                         shortcut_type="B", bias_downsample=False).eval().to(device)
    for p in net.parameters(): p.requires_grad_(False)
    def embed(sid):
        v = clip_ct(extract_roi(sid)["img"]); v = (v - v.mean()) / (v.std() + 1e-6)
        t = torch.from_numpy(np.ascontiguousarray(v)).float()[None, None]
        t = torch.nn.functional.interpolate(t, size=(64,128,128), mode="trilinear", align_corners=False).to(device)
        with torch.no_grad(): return net(t).squeeze().cpu().numpy().astype(np.float32)
    return embed

def build_merlin(device):
    from merlin import Merlin
    from merlin.data.monai_transforms import ImageTransforms
    model = Merlin(ImageEmbedding=True).eval().to(device)
    for p in model.parameters(): p.requires_grad_(False)
    def embed(sid):                                   # Merlin vuole il VOLUME INTERO
        data = ImageTransforms({"image": str(IMAGES_DIR / f"{sid}_0000.nii.gz")})
        img = data["image"].unsqueeze(0).to(device).float()
        with torch.no_grad(): return model(img).squeeze().detach().cpu().numpy().astype(np.float32)
    return embed

def build_ctfm(device):
    from lighter_zoo import SegResEncoder
    model = SegResEncoder.from_pretrained("project-lighter/ct_fm_feature_extractor").eval().to(device)
    for p in model.parameters(): p.requires_grad_(False)
    def embed(sid):
        v = np.clip(extract_roi(sid)["img"], -1024, 2048); v = (v + 1024.0) / 3072.0
        t = torch.from_numpy(np.ascontiguousarray(v)).float()[None, None]
        t = torch.nn.functional.interpolate(t, size=(128,128,128), mode="trilinear", align_corners=False).to(device)
        with torch.no_grad(): out = model(t)
        deep = out[-1] if isinstance(out, (list, tuple)) else out
        return deep.mean(dim=(2,3,4)).squeeze().detach().cpu().numpy().astype(np.float32)
    return embed

def build_genesis(device):
    p = hf_hub_download(repo_id="MrGiovanni/ModelsGenesis", filename="Genesis_Chest_CT.pt")
    ck = torch.load(p, map_location="cpu", weights_only=False); sd = ck.get("state_dict", ck)
    model = UNet3D(); model.load_state_dict({k.replace("module.", ""): v for k, v in sd.items()}, strict=False)
    model.eval().to(device)
    for pp in model.parameters(): pp.requires_grad_(False)
    def embed(sid):
        v = np.clip(extract_roi(sid)["img"], -1000, 1000); v = (v + 1000.0) / 2000.0
        t = torch.from_numpy(np.ascontiguousarray(v)).float()[None, None]
        t = torch.nn.functional.interpolate(t, size=(64,128,128), mode="trilinear", align_corners=False).to(device)
        with torch.no_grad():
            o64, _ = model.down_tr64(t); o128, _ = model.down_tr128(o64)
            o256, _ = model.down_tr256(o128); o512, _ = model.down_tr512(o256)
            return o512.mean(dim=(2,3,4)).squeeze().detach().cpu().numpy().astype(np.float32)
    return embed

def build_stunet(device):
    wpath = PROJECT_DIR / "ml_study" / "model_weights" / "STU-Net-B.pth"
    if not wpath.exists():
        wpath.parent.mkdir(parents=True, exist_ok=True)
        import gdown
        gdown.download("https://drive.google.com/uc?id=1BHCp1Ort-OaVFwaZmvsG4qHiKiPeNb4h", str(wpath), quiet=False)
    model = STUNet(1, 105, depth=[1]*6, dims=[32,64,128,256,512,512],
                   pool_op_kernel_sizes=[[2,2,2]]*5, conv_kernel_sizes=[[3,3,3]]*6)
    ck = torch.load(wpath, map_location="cpu", weights_only=False)
    model.load_state_dict(ck.get("state_dict", ck), strict=False); model.eval().to(device)
    for pp in model.parameters(): pp.requires_grad_(False)
    def embed(sid):
        v = np.clip(extract_roi(sid)["img"], -1000, 1000); v = (v + 1000.0) / 2000.0
        t = torch.from_numpy(np.ascontiguousarray(v)).float()[None, None]
        t = torch.nn.functional.interpolate(t, size=(128,128,128), mode="trilinear", align_corners=False).to(device)
        with torch.no_grad(): return model.encode(t).mean(dim=(2,3,4)).squeeze().detach().cpu().numpy().astype(np.float32)
    return embed
print("Loader dei 7 modelli pronti.")

Loader dei 7 modelli pronti.


## 5) Estrazione di tutti i modelli (resumabile)
Scarica i pesi (HuggingFace / Google Drive) la prima volta, poi estrae e salva un parquet per
modello. Se un modello non è disponibile, viene **saltato** senza bloccare gli altri.

In [8]:
BUILDERS = {
    "imagenet": build_imagenet, "radimagenet": build_radimagenet, "medicalnet": build_medicalnet,
    "merlin": build_merlin, "ctfm": build_ctfm, "models_genesis": build_genesis, "stu_net": build_stunet,
}
def estrai(nome, build_fn):
    out = CACHE_FEAT / f"feat_{nome}.parquet"
    prev = pd.read_parquet(out) if out.exists() else None
    done = set(prev["study_id"]) if prev is not None else set()
    todo = [s for s in cohort["study_id"] if s not in done]
    if not todo:
        print(f"  {nome}: già completo ({len(done)} casi)."); return
    try:
        embed = build_fn(DEVICE)
    except Exception as e:
        print(f"  ⚠️ {nome}: NON disponibile ({type(e).__name__}: {str(e)[:80]}) -> saltato"); return
    print(f"  {nome}: estraggo {len(todo)} casi su {DEVICE} ...")
    rows = []
    for i, sid in enumerate(todo, 1):
        try:
            emb = embed(sid); rec = {f"{nome}_{j}": float(v) for j, v in enumerate(emb)}
        except Exception as e:
            print(f"    ⚠️ {sid}: {type(e).__name__} -> NaN"); rec = {}
        rec["study_id"] = sid; rows.append(rec)
        if i % 20 == 0 or i == len(todo):
            cur = pd.DataFrame(rows)
            (pd.concat([prev, cur], ignore_index=True).drop_duplicates("study_id", keep="last")
             if prev is not None else cur).to_parquet(out, index=False)
            print(f"    {i}/{len(todo)}")

for nome, fn in BUILDERS.items():
    estrai(nome, fn)
print("\\n✅ Estrazione deep completata.")
for nome in BUILDERS:
    p = CACHE_FEAT / f"feat_{nome}.parquet"
    if p.exists():
        d = pd.read_parquet(p); print(f"  {nome:16s}: {d.shape[1]-1:4d} numeri × {len(d)} casi")

  imagenet: già completo (326 casi).


  radimagenet: già completo (326 casi).
  medicalnet: già completo (326 casi).
  merlin: già completo (326 casi).


  ctfm: già completo (326 casi).


  models_genesis: già completo (326 casi).
  stu_net: già completo (326 casi).
\n✅ Estrazione deep completata.
  imagenet        : 2048 numeri × 326 casi
  radimagenet     : 2048 numeri × 326 casi


  medicalnet      : 2048 numeri × 326 casi
  merlin          : 2048 numeri × 326 casi
  ctfm            :  512 numeri × 326 casi
  models_genesis  :  512 numeri × 326 casi
  stu_net         :  512 numeri × 326 casi
